# Week 2 exercise: Your technical tutor, streaming, with a model switch

**Lesson:** [week2/week2 EXERCISE.ipynb](../../../week2/week2%20EXERCISE.ipynb), *A full prototype of your question answerer*. **About 40 minutes.**

**What you practice here**

- Expertise through the system prompt: a choice of system prompts, one picked for each question.
- A model switch across providers: a menu of clients and models built from your keys, with Ollama's local model always on it.
- Streaming into a `gr.Chatbot`: a Chat Completions call with streaming on, yielding the whole chat as the answer grows.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week2/week2_exercise", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The Week 2 exercise asks you to turn your Week 1 technical tutor into a full prototype: a Gradio UI, streaming, a
system prompt that adds expertise, and a switch between models. Build the parts behind such a page. One dropdown picks
the expertise, which picks the system prompt. Another picks the model, from a menu of clients for each provider you
have a key for. The answer streams into a `gr.Chatbot`, which shows the whole conversation, so your function yields
the whole chat each time, as Day 5's callbacks return it.

`model_menu` makes the clients, and `answer` takes the menu as its first parameter and makes the Chat Completions
call itself. The checks hand it stand-ins shaped like the real client, which stream canned answers in short chunks,
so nothing is spent. The `gr.Blocks` page itself goes in Try it for real.

## What each call should give

```python
menu = model_menu({"OPENAI_API_KEY": "sk-proj-..."})
list(menu)          # ['GPT-4o mini', 'Llama 3.2']
menu["Llama 3.2"]   # (an OpenAI client at http://localhost:11434/v1 with the key 'ollama', 'llama3.2')

build_messages("What does yield from do?", [], "python")
# [{'role': 'system', 'content': EXPERTISE['python']},
#  {'role': 'user', 'content': 'What does yield from do?'}]

build_messages("What does yield from do?", [], "cooking")
# ValueError: Unknown expertise 'cooking'. Choose from: python, llm engineering

for chat in answer(menu, "GPT-4o mini", "What does yield from do?", [], "python"):
    print(chat[-1])
# makes one request with streaming on, and yields the whole chat each time, ending with the answer so far:
# {'role': 'assistant', 'content': ''}
# {'role': 'assistant', 'content': 'yield from'}
# {'role': 'assistant', 'content': 'yield from hands each value'}  ... and so on
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
from openai import OpenAI

### 1 · `EXPERTISE`

The tutor's expertise: a dict from an expertise's name to a system prompt that gives the tutor that expertise. It has
at least two entries, such as `"python"` for a patient Python tutor and `"llm engineering"` for an expert in building
on LLMs. Each prompt is different, and a string of 20 characters or more.

<details><summary>Hint</summary>

```python
EXPERTISE = {"python": "You are a patient Python tutor. ...",
             "llm engineering": "You are an expert LLM engineer. ..."}
```

</details>

In [ ]:
EXPERTISE = {}

In [ ]:
check("EXPERTISE")

### 2 · `model_menu(env)`

Builds the model switch: a dict from a dropdown label to a `(client, model)` pair. `env` is a mapping of environment
variables, such as `os.environ` after `load_dotenv(override=True)`: read the keys from it only. The menu holds:

- `"GPT-4o mini"`, when `OPENAI_API_KEY` is set: a client with that key, and the model `"gpt-4o-mini"`, which your
  Week 1 tutor used;
- `"Claude Haiku"`, when `ANTHROPIC_API_KEY` is set: a client at Anthropic's endpoint, `https://api.anthropic.com/v1/`,
  with that key, and the model `"claude-haiku-4-5"`;
- `"Llama 3.2"`, always: a client at Ollama's endpoint, `http://localhost:11434/v1`, with the key `"ollama"`, and the
  model `"llama3.2"`.

Leave out an entry whose key is missing or empty. Making a client sends nothing, so the checks use the real `OpenAI`
here.

<details><summary>Hint</summary>

```python
menu["Llama 3.2"] = (OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"), "llama3.2")
```

</details>

In [ ]:
def model_menu(env):
    # Write model_menu here, as described above.
    raise NotImplementedError("model_menu is not written yet")

In [ ]:
check("model_menu")

### 3 · `build_messages(question, history, expertise)`

Builds the messages for one question, with the chosen expertise. If `expertise` is not a key of `EXPERTISE`, raise
`ValueError` with a message that names every key. Otherwise return a new list: a system message holding that
expertise's prompt, then one message for each history entry, in order, with only its role and content, then a user
message holding `question`. As on Day 3, keep only the role and content, because the history that Gradio passes
carries more keys. Do not change the history.

<details><summary>Hint</summary>

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
return [{"role": "system", "content": EXPERTISE[expertise]}] + history + [{"role": "user", "content": question}]
```

</details>

In [ ]:
def build_messages(question, history, expertise):
    # Write build_messages here, as described above.
    raise NotImplementedError("build_messages is not written yet")

In [ ]:
check("build_messages")

### 4 · `answer(menu, model_name, question, history, expertise)`

Streams the answer into the chat, yielding the whole chat each time: a generator for `gr.Chatbot(type="messages")`.
It:

1. checks the choices before its first yield: if `model_name` is not a key of `menu`, it raises `ValueError`, and it
   builds the messages with `build_messages`, which raises `ValueError` for an unknown expertise. So a wrong choice
   fails at once, and no model is called;
2. yields a new list, the chat: the history's entries (as they are, or with only their role and content), then a user
   message holding `question` and an assistant message with empty content, so that the question shows at once;
3. makes one Chat Completions call with the chosen pair's client and model, the messages from step 1, and streaming
   on;
4. after each chunk, yields a new list: the chat again, with the assistant's content set to all the text so far.

Do not change the history you were given.

<details><summary>Hint</summary>

```python
client, model = menu[model_name]
stream = client.chat.completions.create(model=model, messages=messages, stream=True)
for chunk in stream:
    text += chunk.choices[0].delta.content or ""
    yield chat + [{"role": "assistant", "content": text}]
```

</details>

In [ ]:
def answer(menu, model_name, question, history, expertise):
    # Write answer here, as described above.
    raise NotImplementedError("answer is not written yet")

In [ ]:
check("answer")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

An answer from `gpt-4o-mini` costs a fraction of a cent. The Llama choice needs Ollama running, with `llama3.2` pulled.

1. Load your keys with `load_dotenv(override=True)`, and build `menu = model_menu(os.environ)`.
2. Write a generator function `respond(question, history, model_name, expertise)` that does
   `yield from answer(menu, model_name, question, history, expertise)`.
3. In `gr.Blocks`, after `import gradio as gr`, add a row with a model `gr.Dropdown(list(menu))` and an expertise
   `gr.Dropdown(list(EXPERTISE))`, then a `gr.Chatbot(type="messages")` and a `gr.Textbox`. When the textbox is
   submitted, run `respond` with the chatbot as its output, then clear the textbox.
4. Ask your Week 1 question about `yield from`, then switch models halfway and watch the new model carry on the
   conversation.
5. Optional, for the bonus: give the tutor a tool, as on Day 4, on a first call without streaming; or read each answer
   aloud with Day 5's `talker`.

<details><summary>Stuck? One way to write it</summary>

```python
import os

import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)
menu = model_menu(os.environ)


def respond(question, history, model_name, expertise):  # a generator function, so the chat streams
    yield from answer(menu, model_name, question, history, expertise)


with gr.Blocks(title="Technical tutor") as ui:
    with gr.Row():
        model = gr.Dropdown(list(menu), value=list(menu)[0], label="Model")
        expertise = gr.Dropdown(list(EXPERTISE), value=list(EXPERTISE)[0], label="Expertise")
    chatbot = gr.Chatbot(type="messages", height=450)
    question = gr.Textbox(label="Your question:")
    question.submit(respond, inputs=[question, chatbot, model, expertise], outputs=chatbot).then(
        lambda: "", outputs=question)

ui.launch()
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.